# P3-v2 read-only post-primary analysis

Run on CPU only after **both** frozen backbones have completed 18/18 full units. This notebook downloads no checkpoint and performs no model inference. It verifies the private arrays and manifests against the frozen source/construct replay, computes all twelve descriptive cells and source-ID-clustered intervals, and writes a private aggregate report plus a source-ID-free CSV. It never modifies P1/P2/P3-v1 or the 36 prediction units. Lower-link SQL remains explicitly unavailable because lower-link quantiles were not archived; it is not imputed from medians. Analysis runs in a fresh Python subprocess so reruns use the pulled code even if this Colab kernel had imported an older version.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, subprocess, sys

REPO = Path('/content/tsfm-covariate-faithfulness')
REPO_URL = 'https://github.com/FlyMe2star/tsfm-covariate-faithfulness.git'
if REPO.exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', 'main', REPO_URL, str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REPO / 'requirements/p3-preflight.txt')], cwd=REPO, check=True)
src_path = str(REPO / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Analysis code commit:', subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
DATA_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_source_data')
ARCHIVE_ROOT = Path('/content/drive/MyDrive/tsfm-covariate-faithfulness/p3_v2_model_v1')
command = [sys.executable, '-m', 'covfaith_p3_v2_analysis.cli', '--repo', str(REPO), '--data-root', str(DATA_ROOT), '--archive-root', str(ARCHIVE_ROOT)]
completed = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
print(completed.stdout)
if completed.returncode:
    diagnostic = next((line for line in completed.stderr.splitlines() if line.startswith('P3_V2_ANALYSIS_ERROR:')), None)
    if diagnostic is None:
        diagnostic = completed.stderr.strip().splitlines()[-1] if completed.stderr.strip() else 'Analysis subprocess failed without stderr.'
    raise RuntimeError(diagnostic)
